## Conversational AI (26/27) Exercise 2: Fine-tuning for Task-aligned Response Generation

When asking task-specific questions, an off-the-shelf language model may not have the necessary domain knowledge to provide adequate answers. Fine-tuning the model on question–answer pairs from the relevant domain can help address this gap.

In this exercise, our goal is to fine-tune a large language model (Qwen3), aiming to handle task-oriented conversational modeling more effectively.
We will primarily focus on **Supervised Fine-Tuning (SFT)** ([link](https://huggingface.co/learn/llm-course/en/chapter11/1)), where we train Qwen3 using reference responses from our dataset. At the end of the notebook, we will also introduce **Direct Preference Optimization (DPO)** ([link](https://huggingface.co/blog/ariG23498/rlhf-to-dpo)) as an additional learning exercise. This section will demonstrate another approach to fine-tuning language models.

Reminder: we  focus on DSTC11 Task5, subtask 3 **Knowledge-grounded Response Generation**. The task and dataset are explained in detail [here](https://github.com/alexa/dstc11-track5).
In essence, the task is to respond to subjective user requests regarding hotels and restaurants, e.g. "does the restaurant have a nice vibe?". Our conversational model has to generate a response to this question based on conversation history.


To fine-tune our model, we will use LoRA fine-tuning.
LoRA (Low-Rank Adaptation of Large Language Models) allows us to fine-tune models more efficiently by updating only a small subset of parameters instead of modifying all of the model’s weights. You can read more [here](https://arxiv.org/abs/2106.09685) and [here](https://huggingface.co/learn/llm-course/en/chapter11/4).

**This notebook contains the following parts:**
1. Formatting the DSTC11 Task5 data in the proper structure
2. Fine-tuning Qwen3 on the DSTC11 Task5 data
3. Analyzing the model behavior in comparison to its off-the-shelf baseline

## Exercise

The goal is to compare the performance between off-the-shelf usage and a domain-specific fine-tuned model. For this exercise, we will focus on manual human evaluation. 

Throughout the notebook, you will find questions you need to answer to complete the exercise. These are both coding questions and questions that evaluate your understanding of the data, the process, and the model behavior. These questions are indicated as  **<span style="background:yellow">Q#</span>**.

**<span style="color:red">Important Note:</span>** Fine-tuning a model is time-consuming. Since this exercise requires you to experiment with different parameters, completing it may take longer than expected. Therefore, please start the exercise as early as possible and avoid leaving it until the last day.


**Exercise steps:**
1. Load the dataset and convert it to the appropriate format. **(Q1)**
2. Prepare your data and fine-tune your model on our dataset. **(Q2)**
3. Manually compare the off-the-shelf vs finetuned Qwen3 performance on 10 dataset samples. Answer questions about the exercise, based on your coding steps and manual analysis. **(Q3 - Q9)**

## Reminders about Kaggle notebooks
1. Pay attention to usage statistics, especially memory, CPU, and GPU
2. Pay attention to the quota of GPU (measured in hours)
3. "Turn OFF" the internet after each session. Turn on the internet when starting a session.
4. "Turn off" the accelerator after each use. __Turn the accelerator on when starting a session__.
5. Save a version after you make changes. This ensures that your teammate can see the latest changes. If you get a question from Kaggle about versions, you can revert to the latest version. With "quick save" you can save a version without running everything.
6. Some blocks of codes can take longer to run. Read the instructions and be patient! 

## Installation

The Kaggle notebooks use a Python 3 environment, and they are already "pre-loaded" with various analytic Python packages, like Json, Pandas, and Numpy. If you are curious, you can see the package definition in [this repository](https://github.com/kaggle/docker-python).

We will install and load relevant packages for working with language models: Transformers, TRL (short for Transformer Reinforcement Learning), and PEFT (short for Parameter-Efficient Fine-Tuning).

This is a short explanation regarding each library:
- **Transformers**: We use this library to load and work with our base model.
- **TRL**: We use this library to fine-tune our base model.
- **PEFT**: We use this library to specify which parameters to update instead of updating all of them, resulting in more lightweight fine-tuning.

In [ ]:
!pip install transformers==4.57.1
!pip install trl==0.24.0
!pip install peft==0.16.0
!pip install --force-reinstall "numpy==1.26.4" "pyarrow==17.0.0"

In [ ]:
# Check versions
import transformers, trl, peft
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)

## Imports

The following code loads several standard packages, packages for working with LLMs, packages for defining our parameters, and also packages for training the model.

A brief explanation of the functions that we loaded:

- **AutoModelForCausalLM, AutoTokenizer**: To load our base language model and the tokenizer.
- **SFTTrainer, SFTConfig**: To define training configurations and create a trainer object for supervised fine-tuning.
- **LoraConfig**: Specifies the LoRA parameters we want to use for parameter-efficient fine-tuning.
- **get_peft_model**: Creates the adapter model by combining our base model with the LoRA configuration, which is then passed to the trainer.
- **PeftModel**: Used to load or save the fine-tuned model after training.

Also, as you can see, we set our seed to a specific number. The reason for this is to make our fine-tuning process reproducible. This ensures that aspects like data shuffling, weight initialization, and other random operations remain consistent across different runs. You can experiment with different seed values if you are interested in seeing how they affect the results.


In [ ]:
import numpy as np 
import json
import os
import shutil
import subprocess
import sys
from typing import List
from datasets import Dataset
import torch
import random

torch.manual_seed(3407); random.seed(3407); np.random.seed(3407)


from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, PeftModel

## Clone the data/task repository

We start by cloning the data and task repository and changing the working directory to that directory.

In [ ]:
def setup_repo(repo_url: str, repo_name: str, work_dir: str = "/kaggle/working"):
    os.chdir(work_dir)
    
    # Remove repo if it exists
    if os.path.exists(os.path.join(work_dir, repo_name)):
        shutil.rmtree(os.path.join(work_dir, repo_name))
    
    # Clone repo
    subprocess.run(["git", "clone", repo_url], check=True)
    
    # Move into repo/data
    os.chdir(os.path.join(repo_name, "data"))


setup_repo("https://github.com/lkra/dstc11-track5.git", "dstc11-track5")

## Exploring the data
The data we will use contains these essential components: 
- __knowledge__: The reviews for the corresponding restaurant or hotel (the data also contains FAQs - we won't be using those!)
- __dialogue__: The dialogue history between the user and the system
- __response__: The ground-truth response we expect from the system

Let's list all files in the current directory iteratively and then load the data first to understand it better!

In [ ]:
## List all files in the current directory iteratively:
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
with open('train/logs.json', 'r') as f:
    train_ds=json.load(f)

with open('train/labels.json', 'r') as f:
    labels=json.load(f)

with open('knowledge.json', 'r') as f:
    knowledge_base=json.load(f)

## Create Dataset
Until now, we have looked at and applied our functions to only one sample. Later, when we want to fine-tune our model, it is thus useful to convert our dataset into a HuggingFace dataset. Converting our dataset into a HuggingFace Dataset provides several advantages, such as faster data loading and processing. You can read more about this [here](https://huggingface.co/docs/datasets/en/create_dataset).

<span style="background:yellow">__Q1:__ We would like to load the data and create a validation set and a test set. For this, we would have a code similar to **exercise 1**, ending with a function that takes as input the split of the DSTC11 dataset and returns the dataset processed by all the steps we applied to the training dataset.</span>

* **format_dialogue:** Formats a list of dialogue by turning it into a readable list. Given a list of dictionaries, each representing a dialogue turn, this function formats the dialogue by prefixing each turn with either "user" or "system" as the role based on the speaker. The speaker is identified by the 'speaker' key in the dictionary: 'U' for user, else it is the system.

In [ ]:
def format_dialogue(dialogue: List[dict]) -> List[dict]: 
    """
    Args:
    dialogue (List[dict]): A list of dictionaries where each dictionary contains two keys:
        - 'speaker' (str): A string indicating the speaker of the turn ('U' for user, 'S' for system).
        - 'text' (str): The text spoken by the respective speaker.

    Returns:
        List[dict]: A new array with a specific role and content

    """
    # Your solution here
    messages=[]
    messages.append({"role": "system", "content": "You are an assistant."})
    for dialogue_element in dialogue:
        role = ##here##
        messages.append({"role": role, "content": ##here##})

    return messages

* **reformat_dataset:** this function formats our dataset into a structure suitable for conversational language-model training. For each sample, it extracts the dialogue and its associated response, appends the response as a message with the "system" role, and adds the resulting message list to a new dataset. Samples that cause errors are skipped. The final output is the Hugging Face Dataset, containing conversations organized as lists of role-content message objects.

NOTE: This function is slightly different from the one in exercise 1. Here, you only need to access the dialogue and the response.


In [ ]:
def reformat_dataset(dataset, labels_dataset): 
    reformatted_dataset = {
        "messages": []
    }
    for sample_index in range(len(dataset)): 
        # Your solution here
        try:
            sample_dialogue = ##here##
            sample_response = ##here##
            sample_dialogue.append({"role": "system", "content": sample_response})
            
            reformatted_dataset["messages"].append(sample_dialogue)
        except:
            continue


        
    return reformatted_dataset

reformatted_dataset = reformat_dataset(train_ds, labels)
dataset = Dataset.from_dict(reformatted_dataset)
dataset

From this point on, we have already created the dataset for our training dataset. In the following function, you need to do the same for the validation and test datasets.

* **process_dataset_split:** based on the input string of the function, load the corresponding split and create the dataset using the same process as for the training data.

In [ ]:
def process_dataset_split(split: str) -> Dataset: 
    """Loads, reformats, and processes a dataset split for model training or evaluation.

    This function loads a dataset split (e.g., 'val', 'test') and generates a dataset for it, similar to what we had for the train split.

    Args:
        split (str): The name of the dataset split to process

    Returns:
        dataset: A HuggingFace `Dataset` object that contains the preprocessed and reformatted data for the specified split.

    """
    # Your solution here.
    

validation_ds = process_dataset_split("val")
test_ds = process_dataset_split("test")

validation_ds, test_ds

## Fine-Tuning

### Setting up

Now that we have our data ready, we can look at fine-tuning our model. We will be using HuggingFace Transformers. 

As a language model, we will be using __Qwen3-1.7B__. Let's load it!

In [ ]:
model_id = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")

What exactly happened in the previous cell?

We did two things.

First, we loaded a tokenizer for the pretrained model Qwen3-1.7B. It is called `AutoTokenizer` since the tokenizer is chosen automatically from the model architecture.

Then, `AutoModelForCausalLM` loads the pretrained Qwen model with a so-called "causal language modeling head" for text generation. By adding a task head to predict the next token, this class suits tasks like creative writing, conversational chat, and code completion. The approach is "autoregressive" because the generation is done one token at a time rather than all at once.
Like AutoTokenizer, AutoModelForCausalLM automatically selects the model architecture from the model class.

### Let's finetune!
For fine-tuning, we use the supervised fine-tuning Trainer from HuggingFace. To understand the function better, you can read the documentation [here](https://huggingface.co/docs/trl/main/en/sft_trainer#trl.SFTTrainer).

The outputs (model weights etc.) will be stored in the _"outputs"_ folder. 

A large part of fine-tuning is determining the values for the **hyperparameters**. Particularly: the batch size, number of epochs, learning rate, and warmup steps are important parameters to get right. 

To prevent our GPU from running out of memory, we will use **gradient accumulation**. This technique allows training neural networks with a larger effective batch size than fits into GPU memory by splitting the batch into smaller mini-batches. With a training batch size of $2$ and gradient accumulation step of $4$ (how many forward and backward passes before updating the model weights), this essentially compares to a batch size of $8$. 

It is up to you to determine the values of the number of training epochs, learning rate, and warmup steps. 
In principle, the recommended standard amount of **epochs** to train is mostly $1-3$ epochs to prevent overfitting. Training for too many epochs can cause the model to memorize the training data rather than generalize to new examples.

The amount of **warmup steps** can range between $3-10\%$ of the total amount of training steps, so this is tied to the amount of epochs you train with. Warmup steps are used to gradually increase the learning rate at the beginning of training. This also allows the optimizer's momentum estimates to stabilize before the learning rate reaches its peak value.

For the **learning rate**, the usual values range between $2e-4$ and $5e-5$. The learning rate determines how much the model's weights are updated during training. A learning rate that is too high can lead to unstable training, while one that is too low can result in slow convergence or insufficient learning. To mitigate these issues, the learning rate is adjusted throughout training using a learning rate scheduler. As previously mentioned, training starts with a warmup phase, where the learning rate gradually increases to its target value to prevent large weight updates and stabilize training. After warmup, **learning rate decay** gradually reduces the learning rate, allowing the model to make smaller updates and converge more smoothly. Common strategies include linear decay, where the learning rate decreases at a constant rate, and cosine decay, where it decreases following a cosine curve.

Play around with these values! The aim is to have a training loss below $1$. A training loss close to $0$ indicates overfitting. 

First, we define the configuration specifying which parts of the model will receive trainable adapter layers. It then wraps the base model with these adapters so that only the small LoRA parameters are fine-tuned while the rest of the model remains frozen.

In [ ]:
peft_cfg = LoraConfig(r=16, 
           lora_alpha=32, 
           lora_dropout = 0.05, 
           bias = "none", 
           use_rslora = False, 
           target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"])

model = get_peft_model(base, peft_cfg)

What exactly happened in the previous cell?

We did two things.

First, we created a LoRA configuration using `LoraConfig`. LoRA is a parameter-efficient fine-tuning (PEFT) technique that allows us to fine-tune a pretrained model without updating all of its parameters. Instead, it introduces small, trainable matrices into selected layers while keeping the original model weights frozen. This significantly reduces the memory and computational resources required for fine-tuning.

The configuration defines several important hyperparameters:

- `r=16` specifies the rank of the LoRA matrices, controlling the number of trainable parameters.
- `lora_alpha=32` controls the scaling of the LoRA updates.
- `lora_dropout=0.05` applies a dropout rate of 5% to the LoRA inputs during training to help prevent overfitting.
- `bias="none"` means that no bias parameters are trained as part of LoRA.
- `use_rslora=False` means that standard LoRA scaling is used instead of rank-stabilized LoRA.
- `target_modules` specifies which layers receive LoRA adapters, including the attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and the feed-forward network projection layers (`gate_proj`, `up_proj`, `down_proj`).

Then, `get_peft_model` applies this LoRA configuration to our pretrained Qwen3-1.7B model. Instead of fine-tuning all the original model weights, we create a model in which only the additional LoRA parameters are trainable. This allows us to adapt the model to our dataset more efficiently while preserving the pretrained weights.

Checks if the GPU supports bfloat16 (bf16).

In [ ]:
def pick_bf16():
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability()
        return major >= 8
    return False

Let’s define our training parameters.

Here, you need to provide values for three of them: NUM_TRAIN_EPOCHS, LEARNING_RATE, and WARMUP_STEPS.

In [ ]:
NUM_TRAIN_EPOCHS = ## your value
LEARNING_RATE    = ## your value
WARMUP_STEPS     = ## your value

os.environ["WANDB_DISABLED"] = "true"

sft_args = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    max_length=1024,
    optim="adamw_torch_fused",
    fp16=not pick_bf16(),
    bf16=pick_bf16(),
    packing=False,
    dataset_num_proc=2,
    report_to="none",
    seed=3407
)


What exactly happened in the previous cell?

First, we disabled Weights & Biases (`WANDB_DISABLED`), an external tool used to track and visualize training experiments. This ensures that our training runs without sending logs to this service.

Then, we created a supervised fine-tuning configuration using `SFTConfig` from Hugging Face's TRL library. This configuration defines the hyperparameters and training settings that will be used to fine-tune our pretrained Qwen model.

The configuration includes several important parameters:

- `output_dir="outputs"` specifies the folder where training outputs, such as model checkpoints, will be saved.
- `per_device_train_batch_size=2` sets the number of training examples processed at once on each device.
- `gradient_accumulation_steps=4` accumulates gradients over four mini-batches before updating the model weights. Combined with a batch size of 2, this results in an effective batch size of 8 on a single GPU.
- `learning_rate=LEARNING_RATE` determines how much the model's trainable weights are updated during training.
- `warmup_steps=WARMUP_STEPS` specifies how many initial training steps are used to gradually increase the learning rate, helping stabilize training.
- `num_train_epochs=NUM_TRAIN_EPOCHS` determines how many times the model goes through the entire training dataset.
- `logging_steps=10` records training information, such as the loss, every 10 training steps.
- `lr_scheduler_type="linear"` applies a linear learning rate schedule, gradually decreasing the learning rate after the warmup phase.
- `weight_decay=0.01` introduces regularization to help prevent overfitting by discouraging excessively large model weights.
- `max_length=1024` limits the maximum sequence length to 1024 tokens.
- `optim="adamw_torch_fused"` selects a computationally efficient implementation of the AdamW optimizer for updating the trainable parameters.
- `fp16=not pick_bf16()` and `bf16=pick_bf16()` select the appropriate 16-bit floating-point precision based on hardware support, reducing memory usage and potentially speeding up training.
- `packing=False` disables sequence packing, meaning multiple short training examples are not combined into a single sequence.
- `dataset_num_proc=2` uses two processes for dataset preprocessing.
- `report_to="none"` disables reporting training metrics to external experiment-tracking services.
- `seed=3407` sets a random seed to help make the training results reproducible.

Together, these settings control how our model is fine-tuned, including the batch size, learning rate, training duration, memory efficiency, and optimization process.

Importantly, `SFTConfig` only defines the training configuration. The actual fine-tuning process begins when we pass these settings to the trainer and start training.

Let’s define our trainer object using the parameters we set earlier.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,      
    eval_dataset=validation_ds,
    args=sft_args,
    processing_class=tok
)


We initialized a supervised fine-tuning trainer using `SFTTrainer` from Hugging Face's TRL library. This trainer brings together our pretrained model with its LoRA adapters, the training and validation datasets, the tokenizer, and the training configuration.

The trainer takes several important parameters:

- `model=model` specifies the Qwen3-1.7B model with the LoRA adapters that we configured earlier. During fine-tuning, only the trainable LoRA parameters will be updated, while the original pretrained model weights remain frozen.
- `train_dataset=dataset` provides the training dataset that the model will learn from.
- `eval_dataset=validation_ds` provides a separate validation dataset used to evaluate the model's performance on examples it has not been trained on. This helps us monitor how well the model generalizes and detect potential overfitting.
- `args=sft_args` passes the training configuration we defined earlier using `SFTConfig`, including the learning rate, batch size, number of epochs, warmup steps, and optimization settings.
- `processing_class=tok` specifies the tokenizer used to convert the training examples into token IDs that the model can process.

The `SFTTrainer` is specifically designed for supervised fine-tuning of language models. It simplifies the training process by handling tasks such as tokenization, batching, loss calculation, and optimization.

Importantly, **this cell only initializes the trainer; it does not start fine-tuning yet**. The actual training process begins when we call `trainer.train()`, which updates the trainable LoRA parameters based on the training dataset.

Now, let’s train the model. Depending on the number of epochs, the process may take between 2 to 6 hours.

In [ ]:
trainer.train()

Finally, save your trained model in the output/adapter directory.

You can then download it for future use.

In [ ]:
trainer.save_model("outputs/adapter")  
tok.save_pretrained("outputs/adapter")

<span style="background:yellow">__Q2:__ Print the values for the three parameters you defined in your tuning procedure. Use the sft_args object to access the values.</span> 

In [ ]:
# Your coding solution here.
print("NUM_TRAIN_EPOCHS: ", sft_args.##Your answer here##)
print("LEARNING_RATE: ", sft_args.##Your answer here##)
print("WARMUP_STEPS: ", sft_args.##Your answer here##)

#### Now let's take a look at what our fine-tuned model does! 

In [ ]:
base_model_id = "Qwen/Qwen3-1.7B"
adapter_path  = "outputs/adapter" 

Load the base model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
model_base = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype="auto", device_map="auto")
model_base_for_adapter = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype="auto", device_map="auto")

Now, load the finetuned model from outputs/adapter.

In [ ]:
model = PeftModel.from_pretrained(model_base_for_adapter, adapter_path)

We loaded our previously fine-tuned LoRA adapters into the pretrained model using `PeftModel.from_pretrained` from Hugging Face's PEFT library.

Remember that during LoRA fine-tuning, we only trained the additional LoRA parameters while keeping the original pretrained model weights frozen. Therefore, we can save and load these adapters separately without having to save the entire model.

The function takes two important parameters:

- `model_base_for_adapter` specifies the original pretrained model (Qwen3-1.7B) onto which the fine-tuned LoRA adapters will be loaded. The base model provides the original pretrained weights.
- `adapter_path` specifies the location where our previously trained LoRA adapter weights and configuration are stored.

The `PeftModel.from_pretrained` function loads the saved LoRA adapters and attaches them to the base model. This allows us to use the knowledge acquired during fine-tuning together with the original pretrained model.

Importantly, **this step does not train the model again**. It simply loads the previously trained adapters so that we can use our fine-tuned model for tasks such as text generation and inference.

By default, the loaded adapters are in inference mode, meaning their weights are not set to be trainable. Also, the adapters remain separate from the original model weights unless we explicitly merge them.

Now, let's generate outputs for a dialog using the base model and the finetuned model.

In [ ]:
dialogue = test_ds[0]['messages'][:-1]
response = test_ds[0]['messages'][-1]

dialogue, response

In [ ]:
text = tokenizer.apply_chat_template(dialogue, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = tokenizer([text], return_tensors="pt").to(model_base.device)

generated_ids = model_base.generate(**model_inputs, max_new_tokens=500)
output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

print("Base Model: ", tokenizer.decode(output_ids, skip_special_tokens=True).strip())
print("Ground-truth: ", response["content"])

In [ ]:
text = tokenizer.apply_chat_template(dialogue, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens=500)
output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

print("Finetuned Model: ", tokenizer.decode(output_ids, skip_special_tokens=True).strip())
print("Ground-truth: ", response["content"])

You can find a detailed explanation of the two cells above in Notebook 1.

## Analyze Model Responses
Now that we have our model up and running, let's look at what types of generations the fine-tuned model produces and compare them with the generations of the vanilla model. For this, we will use samples from the test set. Pick the first 10 samples from the __test dataset__ for the analysis and use those to answer the questions below. 

Note: Please print the outputs of these 10 examples.

#### Questions

Please answer each question with a single paragraph, consisting of 2-5 sentences. Include examples to support your answers.


<span style="background:yellow">__Q3:__ What values for the following hyperparameters did you set: number of epochs, learning rate, and warmup steps? Explain briefly why you chose these parameters. </span>


```
# Your answer here
```

<span style="background:yellow">__Q4:__ What is the training loss you achieved, what is its trend over the iterations, and what does this mean for your model? </span>

```
# Your answer here
```

<span style="background:yellow">__Q5:__ Compare the off-the-shelf to the fine-tuned model: What do you notice in terms of quality of the generation? Are the generations responding to the query?</span> 


```
# Your answer here
```

<span style="background:yellow">__Q6:__ Repeat the experiment with a different hyperparameter value for number of epochs, learning rate, or warmup steps. Fine-tune the model using these new settings, and describe your observations regarding the training loss, as well as the quality of the model’s answers to the questions from the test dataset.</span> 


```
# Your answer here
```

<span style="background:yellow">__Q7:__ How does the fine-tuned model perform in the following scenarios? Compare this to your answer about the vanilla model in exercise 1.</span> 
1. When all reviews agree with each other
2. When only one review disagrees
3. When opinions in the reviews are mixed (i.e. high disagreement)

```
# Your answer here
```

<span style="background:yellow">__Q8:__ How does the length of the dialogue/conversation history affect the fine-tuned model's generation? Compare this to your answer about the vanilla model in exercise 1.</span> 

```
# Your answer here
```

<span style="background:yellow">__Q9:__ Create 5 samples that come from a different domain (e.g., airports, shops, web-stores; you can use an LLM or search the web for this). How does the fine-tuned LLM perform on these, compared to the dialogues in the DSTC-11 task5 domains?</span> 

```
# Your answer here
```

# Important: This section is provided for your learning and understanding only. There will be no quiz questions based on this section (DPO).

## Direct Preference Optimization (DPO)

So far, we have fine-tuned our **Qwen3-1.7B** model using Supervised Fine-Tuning (SFT) on the DSTC11 dataset. During SFT, the model learns to predict the reference response given a dialogue history.

However, SFT does not explicitly distinguish between more and less desirable responses to the same dialogue.

**Direct Preference Optimization (DPO)** is an alternative training objective that learns from preferences between responses. Instead of training only on reference responses, DPO uses three elements:

- **Prompt:** The dialogue history provided to the model.
- **Chosen response:** A preferred response to the prompt.
- **Rejected response:** A less desirable response to the same prompt.

DPO encourages the model to assign relatively higher probability to the chosen response than to the rejected response, while comparing it against a fixed reference model.

### Applying DPO to our dataset

Our DSTC11 dataset contains dialogue histories and their corresponding reference responses, but it does not provide explicit preference pairs.

To demonstrate DPO using the same dataset, we will construct **synthetic preference pairs**:

1. Use the dialogue history as the prompt.
2. Use its original reference response as the chosen response.
3. Randomly select a different reference response from the training dataset as the rejected response.

We will then fine-tune our **Qwen3-1.7B base model** using DPO.

**Note:** This is a simplified demonstration of DPO. A response sampled from another dialogue is not necessarily an informative negative example. In practice, DPO works best with meaningful preference comparisons between responses generated for the same prompt, ideally judged by humans or a reliable preference evaluation process.

In [ ]:
model_id = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")

This cell prepares the DSTC11 dataset for DPO. For each dialogue, it creates a pair consisting of:
- Prompt: The dialogue history.
- Chosen response: The original reference response from DSTC11.
- Rejected response: A randomly selected response from another dialogue.

It creates 200 training examples and 40 validation examples, which will be used for DPO fine-tuning. You can increase the number of examples. 

In [ ]:


def make_dpo_dataset(source, n_samples=200, seed=3407):
    rng = random.Random(seed)
    examples = []

    responses = [
        row["messages"][-1]["content"]
        for row in source
    ]

    indices = list(range(len(source)))
    rng.shuffle(indices)

    for i in indices:
        messages = source[i]["messages"]
        chosen = responses[i]

        # Select a different DSTC11 reference response
        candidates = [
            j for j in range(len(responses))
            if j != i and responses[j] != chosen
        ]
        if not candidates:
            continue

        rejected = responses[rng.choice(candidates)]

        dialogue = [
            {
                "role": m["role"],
                "content": m["content"]
            }
            for m in messages[:-1]
        ]

        prompt = tok.apply_chat_template(
            dialogue,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        examples.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected
        })

        if len(examples) >= n_samples:
            break

    return Dataset.from_list(examples)


dpo_train = make_dpo_dataset(dataset, n_samples=200)
dpo_eval = make_dpo_dataset(validation_ds, n_samples=40)

print("DPO training examples:", len(dpo_train))
print("DPO validation examples:", len(dpo_eval))
print("\nExample preference pair:")
print("Chosen:", dpo_train[0]["chosen"])
print("Rejected:", dpo_train[0]["rejected"])


Now let's finetune

In [ ]:
dpo_peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.get_device_capability()[0] >= 8
)

dpo_args = DPOConfig(
    output_dir="outputs/dpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    max_steps=15,
    beta=0.1,
    max_length=1024,
    max_prompt_length=768,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    seed=3407
)

dpo_trainer = DPOTrainer(
    model=base,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tok,
    peft_config=dpo_peft_cfg
)

dpo_trainer.train()
dpo_trainer.save_model("outputs/dpo_adapter")
tok.save_pretrained("outputs/dpo_adapter")

print("DPO adapter saved to outputs/dpo_adapter")

The code and training process are similar to the SFT fine-tuning we covered earlier, so we will not repeat the detailed explanation here.

If you want to train the model for longer, you can increase the value of max_steps.

## Example quiz question from this notebook



**E1: Consider the following `SFTConfig` configuration:**

```python
sft_args = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    max_length=1024,
    optim="adamw_torch_fused",
    fp16=not pick_bf16(),
    bf16=pick_bf16(),
    packing=False,
    dataset_num_proc=2,
    report_to="none",
    seed=3407
)
```

**Which TWO parameters directly control how the learning rate changes throughout training, and what is their role?**

(A) `learning_rate` sets the maximum learning rate, while `weight_decay` determines how quickly it decreases.

(B) `warmup_steps` controls the initial gradual increase in learning rate, while `lr_scheduler_type` determines the scheduling strategy applied throughout training.

(C) `num_train_epochs` controls the maximum learning rate, while `gradient_accumulation_steps` determines the decay strategy.

(D) `learning_rate` determines the warmup duration, while `lr_scheduler_type` controls the optimizer's weight decay.

**Answer: (B)**


**E2: What is the main purpose of LoRA (Low-Rank Adaptation)?**

(A) To improve the model's accuracy by updating all its pretrained weights during fine-tuning.

(B) To reduce the model's size by permanently removing most of its pretrained parameters.

(C) To fine-tune a pretrained model without updating all of its parameters.

(D) To speed up training by updating only the output layer, which contains all the information needed for fine-tuning.

**Answer: (C)**
